[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/scw634919-bfty/ecommerce-data-analytics-portfolio/blob/main/2-customer-segmentation-rfm/notebook/customer_segmentation_rfm.ipynb)

# Customer Segmentation Using RFM Analysis

Analyze customer purchasing behavior using RFM (Recency, Frequency, Monetary) analysis to better understand customer value and segmentation.

**Business Goal**: Identify valuable customers, at-risk customers, and customer purchasing patterns to support targeted marketing and retention strategies.

## 1. Import Libraries

In [1]:
import os

import pandas as pd

## 2. Load Dataset

The data is the UCI Online Retail dataset (a UK online retailer, December 2010 – December 2011), saved as a UTF-8 CSV file. See `data/README.md` for where to download it.

In [2]:
# Data file location: ../data in the GitHub layout (notebook/ folder), the project folder locally
data_dir = "../data" if os.path.basename(os.getcwd()) == "notebook" else "."

df = pd.read_csv(os.path.join(data_dir, "online_retail.csv"))
print(f"{len(df):,} rows, {df.shape[1]} columns")

541,909 rows, 8 columns


## 3. Data Cleaning

Remove rows without a customer ID or product description, plus returns and cancellations (`Quantity <= 0`) and invalid prices (`UnitPrice <= 0`). Then parse invoice dates and create a `Sales` column (`Quantity × UnitPrice`).

In [3]:
df = df.dropna(subset=["CustomerID", "Description"])
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# Dates look like "12/1/10 8:26" (month/day/two-digit year), so the format is set explicitly
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%y %H:%M")
df["Sales"] = df["Quantity"] * df["UnitPrice"]
print(f"{len(df):,} rows after cleaning")

397,884 rows after cleaning


## 4. RFM Calculation

For each customer:
- **Recency** = days since the last purchase, counted from a snapshot date (the day after the last invoice)
- **Frequency** = number of unique invoices
- **Monetary** = total sales

In [4]:
# Snapshot date: the day after the last invoice
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

In [5]:
# RFM table
rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "InvoiceNo": "nunique",
    "Sales": "sum"
}).reset_index()

rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]

## 5. RFM Scoring

Each metric is split into quartiles and scored 1–4 (4 = best). Recency is reversed, because a more recent purchase is better. Frequency is ranked first, because many customers share the same order count.

In [6]:
rfm["R_score"] = pd.qcut(rfm["Recency"], 4, labels=[4, 3, 2, 1])
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4])
rfm["M_score"] = pd.qcut(rfm["Monetary"], 4, labels=[1, 2, 3, 4])

rfm["RFM_Score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

## 6. Customer Segmentation

A simple rule turns the scores into three segments:
- **VIP**: RFM score 444, 443, 434, or 344
- **At Risk**: Recency score 1 or 2 (no recent purchase)
- **Regular**: everyone else

In [7]:
def segment_customer(row):
    if row["RFM_Score"] in ["444", "443", "434", "344"]:
        return "VIP"
    elif row["R_score"] in [1, 2]:
        return "At Risk"
    else:
        return "Regular"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)
print(rfm.head())

   CustomerID  Recency  Frequency  Monetary R_score F_score M_score RFM_Score  \
0     12346.0      326          1  77183.60       1       1       4       114   
1     12347.0        2          7   4310.00       4       4       4       444   
2     12348.0       75          4   1797.24       2       3       4       234   
3     12349.0       19          1   1757.55       3       1       4       314   
4     12350.0      310          1    334.40       1       1       2       112   

   Segment  
0  At Risk  
1      VIP  
2  At Risk  
3  Regular  
4  At Risk  


## 7. Segment Analysis

In [8]:
# Segment-level summary (count, average RFM, total revenue)
segment_summary = rfm.groupby("Segment").agg(
    customer_count=("CustomerID", "count"),
    avg_recency=("Recency", "mean"),
    avg_frequency=("Frequency", "mean"),
    avg_monetary=("Monetary", "mean"),
    total_revenue=("Monetary", "sum")
).reset_index().round(2)
print(segment_summary)

# Each segment's share of customers and revenue
shares = segment_summary.set_index("Segment")[["customer_count", "total_revenue"]]
shares = (shares / shares.sum() * 100).round(1)
shares.columns = ["% of customers", "% of revenue"]
print()
print(shares)

   Segment  customer_count  avg_recency  avg_frequency  avg_monetary  \
0  At Risk            2150       166.33           2.13        840.97   
1  Regular            1311        24.48           2.57        914.40   
2      VIP             877        13.37          12.07       6732.67   

   total_revenue  
0     1808079.53  
1     1198773.21  
2     5904555.16  

         % of customers  % of revenue
Segment                              
At Risk            49.6          20.3
Regular            30.2          13.5
VIP                20.2          66.3


## 8. Key Insights & Business Recommendations

**Key insights**
- **VIP** customers are about **20%** of customers (877) but bring in about **66%** of revenue (£5.90M). On average they last bought 13 days before the snapshot date and placed about 12 orders.
- **At Risk** is the largest group: about **50%** of customers (2,150), with an average of 166 days since their last purchase. They still account for about **20%** of revenue (£1.81M), so winning some of them back is worth the effort.
- **Regular** customers (1,311) bought recently (24 days on average) but only about 2.6 times, so there is room to raise their purchase frequency.

**Business recommendations**
- **VIP:** loyalty rewards and early access to new products. Avoid heavy discounts, since they already buy often.
- **At Risk:** win-back emails with a limited-time offer or new arrivals.
- **Regular:** personalized recommendations, bundles, and cross-sell offers to raise purchase frequency.

## 9. Export CSVs for Tableau Dashboard

Export the segmentation results to CSV files for the Tableau dashboard.

In [9]:
# Save CSVs in the project's outputs folder:
# "../outputs" when the notebook sits in a notebook/ folder (GitHub layout),
# "outputs" when it sits in the project folder itself (local layout)
output_dir = "../outputs" if os.path.basename(os.getcwd()) == "notebook" else "outputs"
os.makedirs(output_dir, exist_ok=True)

# Full RFM table with segment labels
rfm.to_csv(os.path.join(output_dir, "rfm_segments.csv"), index=False)

# Segment-level summary (count, avg RFM, total revenue)
segment_summary.to_csv(os.path.join(output_dir, "segment_summary.csv"), index=False)

print(f"Saved to {output_dir}/:")
print("  rfm_segments.csv -", len(rfm), "customers")
print("  segment_summary.csv -", len(segment_summary), "segments")

Saved to ../outputs/:
  rfm_segments.csv - 4338 customers
  segment_summary.csv - 3 segments
